# Auto waiting

The playwright auto waiting feature ensures that the element is **actionable** before it tries to perform any operation on the element. That is, before performing any action on an element, the playwright does the following checks by default:
* The element should be attached to the DOM.
* The element should be visible, enabled, and stable.
* The element should be clickable, that is, it should receive pointer events.

**If all checks passed, that means the element is actionable** and the playwright can perform the operation on that element. But if any of the conditions are not satisfied, then the playwright will by default wait for `30000 ms` for the condition to be satisfied and the element to be actionable. Still, even after `30000 ms`, if the element is not actionable, then playwright will throw `TimeoutError: Timeout 30000ms exceeded`.

**Example**:

```python
from playwright.sync_api import sync_playwright

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    page = browser.new_page()
    page.goto("https://bootswatch.com/default/") # By default, playwright waits for page loading

    button = page.get_by_label("Button 1")
    
    button.click()
    # Playwright auto-waiting feature kicks in and by default waits for 30000ms for the element to become actionable.
    
    button.click(timeout=2_000)
    # Wait for 2000ms instead of 30000ms, for the element to be actionable
    
    button.click(force=True)                    
    # Perform click, bypassing actionability checking.
    # iff the element is not visible, the click operation will fail immediately without waiting for the default 30000ms timeout.

    browser.close()
```

**Forced Click**:
* By default, `force=False` which means playwright will wait for 30000ms for the element to be **actionable** before throwing **TimeoutError**.
* Setting `force=False` means playwright will by the actionability check and tries to perform the operation without waiting for the element to be visible, enabled, stable & clickable.
    * In this case, it will never throw **TimeoutError**. Since the actionability check is bypassed, Playwright will not wait for the element to become actionable.
    * If the element is actionable, then the operation will get performed, otherwise, the operation will fail without any waiting.

# Auto-Waiting Navigation

* Whenever we try to perform any action/operation like click, Playwright performs some checks and **auto-waiting** along with it to perform the actual action. 
* Similarly, Playwright does **auto-waiting** for navigation as well.
* That is, if we try to visit a new page or load a new page, Playwright auto waits for certain events.
* By default, playwright waits for the page to load.

We can also configure the wait event for navigation using the `wait_until` argument of the `goto()` method.

The `goto()` method `wait_until` argument takes the following 4 arguments:
* **`load`**:
    * This event waits until all the resources of the page are fully loaded in the browser, including HTML contents, any files, text, images, icons, animations, videos, etc.
    * This is the default setting.
* **`domcontentloaded`**:
    * This event waits until the DOM content of the page is fully loaded.
    * DOM Contents refers to the HTML document & its elements, text content, & js files only.
    * It doesn't wait for images, icons, animations, videos, etc.
    * Hence, it's faster than the `load` event.
* **`commit`**:
    * This event waits until the HTML response is received from the server for the web request.
    * It doesn't even wait for the DOM Content loading or parsing of the response.
    * Hence, it's faster than the `domcontentloaded` event.
* **`networkidle`**:
    * This event waits until the network is idle, that is, until the network in & out is stopped.
    * That is, it will wait for any content downloading/loading/async processing happening in the background, even after the page is fully loaded.
    * Once everything is loaded completely and no network activity is happening anymore, this event will get triggered.
    * Hence, it's slower than the `load` event.
    * The `networkidle` event has a max timeout of 500ms.
    * If the `networkidle` event is not triggered within 500ms after the `load` event is triggered, then it will throw `TimeoutError`.

**Example:**

```python
from playwright.sync_api import sync_playwright
from time import perf_counter

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    page = browser.new_page()

    print("page loading ...")
    start = perf_counter()

    page.goto(
        "https://bootswatch.com/default/",
        wait_until= 'networkidle'  # ["commit", "domcontentloaded", "load", "networkidle"]
    )

    time_taken = perf_counter() - start
    print("page loaaded in {} secs".format(round(time_taken,2)))

    browser.close()
```

# Custom Waiting

We'll learn how we can wait for elements or any other locator with the `wait_for()` method, which is also known as **Custom Wait**.

```python
from playwright.sync_api import sync_playwright
from time import perf_counter

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    page = browser.new_page()
    page.goto("http://scrapethissite.com/pages/ajax-javascript/")

    link = page.get_by_role('link',name='2015')
    link.click()

    print("Loading Oscar movies for 2015 ...")
    start = perf_counter()

    first_table_data = page.locator("td.film-title").first
    first_table_data.wait_for() # Default timeout = 30000ms
    
    first_table_data.wait_for(timeout=2_000) # Timeout sets to 2000ms

    first_table_data.wait_for(timeout=2_000, state='visible') #  state = ["attached", "detached", "hidden", "visible"]

    # Altenative option
    page.wait_for_selector(selector="td.film-title")

    time_taken = round(perf_counter() - start,2)
    print("Loaded Oscar movies for 2015 in {} secs".format(time_taken))

    browser.close()
```

# Event Listener

We've learnt how we can wait until some events occur when navigation happens, that is, we can wait:
* until the page loads in.
* until the DOM contents load in, etc.

Now we'll learn how we can listen to these events; that is, whenever an event occurs, we will go ahead and perform some functions.
1. Define the event handler function. 
2. Register the event listener (**before the action that generates the event**), which will listen to the event and call the event handler once the event is triggered.
3. Unregister the event listener.

We can **register the event listener** using the `on()` method, that is, `page.on("event_name", event_handler_function_name)`.

To **unregister the event listener** using the `remove_listener()` method, that is, `page.remove_listener("event_name", event_handler_function_name)`.

If we want to listen for an event once, then use the `once()` method instead of the `on()` method. That is, `page.once("event_name", event_handler_function_name)`.

```python
from playwright.sync_api import sync_playwright
from time import perf_counter

def on_load(page):
    print("Page is loaded ", page)

def on_request(request):
    print("REQUEST SENT: ", request)

def on_filechooser(file_chooser):
    print("File chooser opened")
    file_chooser.set_files("file1.txt")

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    page = browser.new_page()

    # Register the event listener (before the action that generates the event)
    # Here whenever 'load' event occurs, the on_load() function will be called to handle the event.
    # Different types of events: load, domcontentloaded, close, response, request, etc.
    page.on("request", on_request)
    page.on("load", on_load)
    page.on("filechooser", on_filechooser)

    # page.once("load", on_load)
    
    page.goto("https://bootswatch.com/default/")

    file_input = page.get_by_label("Default file input example")
    file_input.click()

    # Unregister the event listener
    page.remove_listener("request", on_request)
    page.remove_listener("load", on_load)
    page.remove_listener("filechooser", on_filechooser)

    browser.close()
```


# Handling dialogs

Previously, we've learned how we register event listeners for different events. 

Now we'll learn how to listen/handle dialog events, mainly: 
* **alert**: OK only -> accept().
* **confirm**: OK or CANCEL -> accept() /dismiss()
* **prompt dialog**: prompt input field and OK/CANCEL -> accept(prompt_message) /dismiss()

```python
from playwright.sync_api import sync_playwright
from time import perf_counter

def on_dialog(dialog):
    print("Dialog opened : ", dialog)
    dialog.accept("Playwright is cool")
    print("Dialog closed : ", dialog)
    # dialog.dismiss()

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    page = browser.new_page()
    page.goto("https://testpages.herokuapp.com/styled/alerts/alert-test.html")

    page.on("dialog", on_dialog)

    alert_btn = page.get_by_text("Show alert box")
    alert_btn.click()

    confirm_btn = page.get_by_text("Show confirm box")
    confirm_btn.click()

    prompt_btn = page.get_by_text("Show prompt box")
    prompt_btn.click()

    browser.close()
```

# Download file

By default, Playwright downloads the file in a temporary location, and once the browser is closed, Playwright deletes the file.

To save the download file in a specific location

```python
# Capture  the download window
with page.expect_download() as download_info:
    img.click()

download = download_info.value
download.save_as("beauty.jpg") # file path
```

**Example**:

```python
from playwright.sync_api import sync_playwright
from time import perf_counter

def on_download(download):
    print("Download received")
    download.save_as("beauty.jpg") # file path

with sync_playwright() as playwright:
    browser = playwright.chromium.launch(headless=False, slow_mo=1000)
    page = browser.new_page()
    page.goto("https://unsplash.com/photos/woman-smiles-at-the-beach-during-sunset-xnS3upQYaOk")

    page.on('download',on_download)

    img = page.get_by_role('link', name="Download free")

    # Capture  the download window
    with page.expect_download() as download_info:
        img.click()

    browser.close()
```

# What is Sync and Async API?

Until now, we haven't done any computation or resource-intensive operation, so the program finishes within less than milliseconds. 
* Also, we are **using the output of one statement in the next statement.
* That is, the **execution depends on the output of the previous statements**.
* This is referred to as **sequential** or **synchronous execution**.

**Example**:

```python
arg ='banana'
val = sync_process(arg)
print(val)
```


Now consider resource-intensive tasks that are independent of each other and can be done parallely. 
* That is, the execution of one task is not dependent on the output of the previous tasks.
* So each task can be executed independently and in parallel.
* This is referred to as **asynchronous** or **multithreaded execution**.

**Example**:

```python
ele1, ele2, ele3 = val1, val2, val3
async_process(ele1)
async_process(ele2)
async_process(ele3)
```

# Asynchronous Playwright

The `await` keyword indicates a synchronous operation that needs to be completed before moving to the next statement.

```python
import asyncio
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=False, slow_mo=1000)
        page = await browser.new_page()
        await page.goto("https://playwright.dev/python")

        print(await page.title())

        await browser.close()

asyncio.run(main())
```